## Datos de Aprobación de Crédito (Credit Approval - CRX)

El conjunto de datos CRX contiene solicitudes de tarjetas de crédito del repositorio UCI. Todos los nombres de atributos y valores han sido anonimizados para proteger la confidencialidad. El objetivo es predecir si una solicitud de crédito es aprobada o rechazada.

**Dataset:** [Credit Approval - UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/credit+approval)

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, StandardScaler

## 2. Carga y Selección de Datos

Se carga el archivo `.data` sin encabezado y se asignan los nombres de columnas manualmente. Los atributos están anonimizados. Los valores faltantes están representados como `'?'`.

| Columna (índice) | Nombre  | Tipo        | Descripción |
|------------------|---------|-------------|-------------|
| 0                | `A1`    | Categórico  | Atributo binario (a, b) |
| 1                | `A2`    | Numérico    | Atributo continuo |
| 2                | `A3`    | Numérico    | Atributo continuo |
| 3                | `A4`    | Categórico  | Atributo multi-clase (u, y, l) |
| 4                | `A5`    | Categórico  | Atributo multi-clase (g, p, gg) |
| 5                | `A6`    | Categórico  | Atributo multi-clase (14 categorías) |
| 6                | `A7`    | Categórico  | Atributo multi-clase (9 categorías) |
| 7                | `A8`    | Numérico    | Atributo continuo |
| 8                | `A9`    | Categórico  | Atributo binario (t, f) |
| 9                | `A10`   | Categórico  | Atributo binario (t, f) |
| 10               | `A11`   | Numérico    | Atributo entero |
| 11               | `A12`   | Categórico  | Atributo binario (t, f) |
| 12               | `A13`   | Categórico  | Atributo multi-clase (g, s, p) |
| 13               | `A14`   | Numérico    | Atributo continuo |
| 14               | `A15`   | Numérico    | Atributo entero |
| 15               | `A16`   | Categórico  | **Variable objetivo:** `+` (aprobado) o `-` (rechazado) |

**Variable objetivo (col. 15):** `A16` (decisión de crédito: `+` aprobado / `-` rechazado)

In [2]:
# Nombres de columnas (el archivo no tiene encabezado)
column_names = ['A1','A2','A3','A4','A5','A6','A7','A8','A9','A10','A11','A12','A13','A14','A15','A16']

dt = pd.read_csv(
    '../Database/PP6_crx.data',
    header=None,
    names=column_names,
    na_values='?',        # los faltantes vienen como '?'
    skipinitialspace=True # elimina espacios al inicio de cada campo
)

# Limpiar espacios en nombres de columnas
dt.columns = dt.columns.str.strip()

# Este dataset no tiene columnas a excluir: todas son relevantes
print(dt.columns.tolist())
# print(pd.isnull(dt).sum())

['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A14', 'A15', 'A16']


In [3]:
numeric_cols = dt.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = dt.select_dtypes(include=['object']).columns.tolist()

In [4]:
# Imputación de columnas numéricas con la mediana (antes de construir KNN)
for col in numeric_cols:
    n_faltantes = dt[col].isna().sum()
    if n_faltantes > 0:
        mediana = dt[col].median()
        dt[col].fillna(mediana, inplace=True)
        print(f"  [{col}] → {n_faltantes} faltantes imputados con mediana={mediana}")

# Escalar columnas numéricas para KNN (distancias comparables)
scaler = StandardScaler()
X_num = scaler.fit_transform(dt[numeric_cols])

# KNN sobre las columnas numéricas (k=5 vecinos)
knn = NearestNeighbors(n_neighbors=6, metric='euclidean')  # 6 porque incluye la propia fila
knn.fit(X_num)
distancias, indices = knn.kneighbors(X_num)  # precalcular vecinos para todas las filas

print(f"Columnas categóricas a procesar: {len(categorical_cols)}")
print(categorical_cols)

# Imputar cada columna categórica con KNN-moda
for col in categorical_cols:
    mask_na = (dt[col] == '?') | (dt[col].isna())
    n_faltantes = mask_na.sum()

    if n_faltantes == 0:
        pass  # sin faltantes, saltar directo a encodear
    else:
        print(f"  [{col}] → {n_faltantes} faltantes, imputando con KNN-moda...")
        filas_na = dt.index[mask_na].tolist()

        for fila in filas_na:
            # Los 6 vecinos incluyen la fila misma en índice 0 → usamos [1:6]
            vecinos_idx = indices[fila][1:6]
            # Tomar los valores de esos vecinos en esta columna (excluir NA)
            valores_vecinos = dt[col].iloc[vecinos_idx]
            valores_vecinos = valores_vecinos[
                ~((valores_vecinos == '?') | valores_vecinos.isna())
            ]

            if len(valores_vecinos) > 0:
                moda = valores_vecinos.mode()[0]  # valor más frecuente entre vecinos
            else:
                # fallback: moda global de la columna (excluyendo NA)
                valores_globales = dt[col][~mask_na]
                moda = valores_globales.mode()[0]

            dt.at[fila, col] = moda

    # Convertir la columna a entero con LabelEncoder
    le = LabelEncoder()
    dt[col] = le.fit_transform(dt[col].astype(str))

print("\n✓ Imputación y codificación completada.")
print(f"¿Quedan '?' en columnas categóricas? {(dt[categorical_cols] == '?').sum().sum()}")
print(f"¿Quedan NaN en todo dt? {dt.isnull().sum().sum()}")
print("\nTipos de datos finales:")
print(dt.dtypes.value_counts())
print("\nPrimeras 5 filas:")
print(dt.head())

  [A2] → 12 faltantes imputados con mediana=28.46
  [A14] → 13 faltantes imputados con mediana=160.0
Columnas categóricas a procesar: 10
['A1', 'A4', 'A5', 'A6', 'A7', 'A9', 'A10', 'A12', 'A13', 'A16']
  [A1] → 12 faltantes, imputando con KNN-moda...
  [A4] → 6 faltantes, imputando con KNN-moda...
  [A5] → 6 faltantes, imputando con KNN-moda...
  [A6] → 9 faltantes, imputando con KNN-moda...
  [A7] → 9 faltantes, imputando con KNN-moda...

✓ Imputación y codificación completada.
¿Quedan '?' en columnas categóricas? 0
¿Quedan NaN en todo dt? 0

Tipos de datos finales:
int64      12
float64     4
Name: count, dtype: int64

Primeras 5 filas:
   A1     A2     A3  A4  A5  A6  A7    A8  A9  A10  A11  A12  A13    A14  A15  \
0   1  30.83  0.000   1   0  12   7  1.25   1    1    1    0    0  202.0    0   
1   0  58.67  4.460   1   0  10   3  3.04   1    1    6    0    0   43.0  560   
2   0  24.50  0.500   1   0  10   3  1.50   1    0    0    0    0  280.0  824   
3   1  27.83  1.540   1   0  

/tmp/ipykernel_19272/3500591629.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dt[col].fillna(mediana, inplace=True)
